# Convert Devang Model 2 To GGUF

Run this in Colab with a GPU runtime. It merges the Devang LoRA adapter into `Qwen/Qwen2.5-3B-Instruct`, converts the merged model to GGUF, and also creates a quantized `Q4_K_M` GGUF for Ollama.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import subprocess

def run_cmd(cmd, shell=True):
    print("\n" + "=" * 80)
    print("Running:")
    print(cmd if isinstance(cmd, str) else " ".join(cmd))
    print("=" * 80)
    subprocess.run(cmd, shell=shell, check=True)

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/devangs_fine_tuned")
ADAPTER_DIR = DRIVE_PROJECT_DIR / "model2_v2_finetuned (1)"
MERGED_DIR = DRIVE_PROJECT_DIR / "merged_devang_model2"
GGUF_F16 = DRIVE_PROJECT_DIR / "devang_model2_f16.gguf"
GGUF_Q4 = DRIVE_PROJECT_DIR / "devang_model2_q4_k_m.gguf"
HF_CACHE_DIR = DRIVE_PROJECT_DIR / "hf_cache"

HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE_DIR / "transformers")

print("Project folder:", DRIVE_PROJECT_DIR)
print("Adapter folder:", ADAPTER_DIR)
print("Merged model folder:", MERGED_DIR)
print("F16 GGUF:", GGUF_F16)
print("Q4 GGUF:", GGUF_Q4)
print("HF cache:", HF_CACHE_DIR)

assert DRIVE_PROJECT_DIR.exists(), f"Missing project folder: {DRIVE_PROJECT_DIR}"
assert ADAPTER_DIR.exists(), f"Missing adapter folder: {ADAPTER_DIR}"

print("\nInstalling uv...")
run_cmd("pip install uv")

print("\nInstalling pinned dependencies with uv...")
run_cmd(
    "uv pip install --system "
    "transformers==4.55.4 "
    "peft==0.17.1 "
    "accelerate==1.10.1 "
    "safetensors "
    "sentencepiece "
    "protobuf "
    "huggingface_hub"
)

print("\nInstalled package versions:")
run_cmd("python -m pip show torch transformers peft accelerate safetensors sentencepiece protobuf huggingface_hub uv")

print("\nImporting libraries...")
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("CUDA is not available. Switch Colab runtime to GPU.")

base_model_id = "Qwen/Qwen2.5-3B-Instruct"

print("\nLoading base model...")
base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("\nLoading Devang fine-tuned adapter...")
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))

print("\nMerging adapter into base model...")
merged = model.merge_and_unload()

print("\nSaving merged Hugging Face model to Drive...")
MERGED_DIR.mkdir(parents=True, exist_ok=True)
merged.save_pretrained(MERGED_DIR, safe_serialization=True)

tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR), use_fast=True)
tokenizer.save_pretrained(MERGED_DIR)

print("Merged model saved:", MERGED_DIR)

print("\nCloning llama.cpp...")
os.chdir("/content")
if not Path("/content/llama.cpp").exists():
    run_cmd("git clone https://github.com/ggerganov/llama.cpp")
else:
    print("llama.cpp already exists.")

print("\nInstalling llama.cpp Python requirements with uv...")
run_cmd("uv pip install --system -r /content/llama.cpp/requirements.txt")

print("\nConverting merged model to F16 GGUF...")
run_cmd(
    [
        "python",
        "/content/llama.cpp/convert_hf_to_gguf.py",
        str(MERGED_DIR),
        "--outfile",
        str(GGUF_F16),
        "--outtype",
        "f16",
    ],
    shell=False,
)

print("\nBuilding llama.cpp quantizer...")
run_cmd("cmake -S /content/llama.cpp -B /content/llama.cpp/build")
run_cmd("cmake --build /content/llama.cpp/build --config Release -j 2")

print("\nQuantizing to Q4_K_M...")
quantizer = Path("/content/llama.cpp/build/bin/llama-quantize")
if not quantizer.exists():
    quantizer = Path("/content/llama.cpp/build/bin/Release/llama-quantize.exe")

assert quantizer.exists(), f"Quantizer not found: {quantizer}"

run_cmd(
    [
        str(quantizer),
        str(GGUF_F16),
        str(GGUF_Q4),
        "Q4_K_M",
    ],
    shell=False,
)

print("\nConversion complete.")
print("F16 GGUF saved at:", GGUF_F16)
print("Quantized GGUF saved at:", GGUF_Q4)
print("Download this file for Ollama:", GGUF_Q4)